# Part 3: ensemble of recurrent location extractors

Flow: **members → probability bundles → recipe search on test-like data → submission → actual LB score**.

* Members are trained elsewhere (`A3_colab.ipynb` / `part2.train`, logged to W&B). Here a member is only a checkpoint plus a loader.
* Every member is run once per split and its O/B/I probabilities are cached as a bundle on Drive. Combinations never retrain anything.
* The recipe (members, weights, threshold) is chosen on `testlike_select` only. `testlike_report`, the 26 organiser examples, checkpoint and calibration are reported for every candidate. Audit is scored once, after freezing.

In [ ]:
# Mount Drive from the Colab sidebar first (Files -> Mount Drive); this notebook never mounts it.
from pathlib import Path
import json, os, re, shutil, subprocess, sys

if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive is not mounted: left sidebar -> folder icon -> "Mount Drive", then rerun this cell')
CANDIDATE_ROOTS = [os.environ.get('A3_ROOT', ''), '/content/drive/MyDrive/KSE_AI301_A3',
                   '/content/drive/.shortcut-targets-by-id/12ao27pY1xBfLeIRZcJYTDFojxW5-88tc/KSE_AI301_A3']
ROOT = next((Path(p) for p in CANDIDATE_ROOTS if p and Path(p).is_dir()), None)
if ROOT is None:
    raise RuntimeError('KSE_AI301_A3 not found. Checked:\n'+'\n'.join(p for p in CANDIDATE_ROOTS if p)
                       +'\nAdd a shortcut to the shared folder in My Drive, or set A3_ROOT to its path')
WORK = Path('/content/a3'); WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK/'A3'                      # fresh clone of the committed code, never a hand-copied file
BRANCH = 'ensemble'
if not (PROJECT/'.git').exists():
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Taburetochka60/A3.git', str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'pull', '-q', '--ff-only'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip()
# A notebook opened from an old copy runs stale cells against new code: compare with the published version.
NOTEBOOK, CELLS = 'ensemble_colab.ipynb', 'c6e203b35336'
published = re.search(r"NOTEBOOK, CELLS = '[^']+', '(\w+)'", (PROJECT/NOTEBOOK).read_text())
if published is None or published.group(1) != CELLS:
    raise RuntimeError(f'This copy of {NOTEBOOK} is out of date. Open the current one: File -> Open notebook -> GitHub, '
                       f'https://colab.research.google.com/github/Taburetochka60/A3/blob/{BRANCH}/{NOTEBOOK}')
sys.path.insert(0, str(PROJECT)); os.chdir(PROJECT)
os.environ['MPLCONFIGDIR'] = '/tmp/a3_matplotlib'

import importlib.metadata, torch
version = importlib.metadata.version('torch')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements-part2.txt')])
assert importlib.metadata.version('torch') == version

from part2_data import file_hash
from part2_compat import pin_grouping_hash
pin_grouping_hash()                      # Colab's Python 3.13 changed ast.dump; keeps the split's code-hash guard valid
SPLIT = PROJECT/'part2_split.json'
ENSEMBLE_SPLIT = PROJECT/'ensemble_split.json'
spec = json.loads(SPLIT.read_text())
DATA = WORK/'data'; DATA.mkdir(exist_ok=True)
DATA_SOURCES = [ROOT/'data', ROOT, ROOT/'domain_shift_source'/'data']
for name, expected in spec['files'].items():      # local copy: much faster to read than Drive
    if not (DATA/name).exists() or file_hash(DATA/name) != expected:
        source = next((d/name for d in DATA_SOURCES if (d/name).is_file()), None)
        if source is None:
            raise RuntimeError(f'{name} not found in any of: {[str(d) for d in DATA_SOURCES]}')
        shutil.copy2(source, DATA/name)
    if file_hash(DATA/name) != expected:
        raise RuntimeError(f'{name} differs from the file part2_split.json was built from')

# The frozen Part 2 checkpoint, found by its recorded hash.
REFERENCE_CANDIDATES = [ROOT/'part2_model'/'checkpoint.pt',
                        ROOT/'results'/'clean_final'/'runs'/'CLEAN_LSTM2_100k_seed42_f362fa37df'/'best.pt']
REFERENCE = next((p for p in REFERENCE_CANDIDATES
                  if p.exists() and file_hash(p) == spec['reference']['checkpoint_sha256']), None)
if REFERENCE is None:
    raise RuntimeError('Frozen Part 2 checkpoint not found under ROOT; add its path to REFERENCE_CANDIDATES')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('root:', ROOT, '\ncode:', BRANCH, COMMIT, '| python', sys.version.split()[0], '| device', DEVICE,
      '\nreference checkpoint:', REFERENCE)
RESULTS = ROOT/'results'/'ensemble'
BUNDLES = RESULTS/'bundles'
# os.environ['WANDB_PROJECT'] = 'ai301-a3'   # same project as the members; needs WANDB_API_KEY

In [ ]:
# One entry per candidate member. `expected` reproduces a score logged for that checkpoint;
# export stops if the bundle disagrees (wrong checkpoint, loader or tokenizer).
MEMBERS = [
    dict(name='word_s42', kind='part2', checkpoint=REFERENCE,
         expected=dict(split='calibration', threshold=.8, policy='strict',
                       f1=spec['reference']['metrics']['calibration']['f1'])),
    # Probe members (ensemble_probe_colab.ipynb) can be added with their checkpoint and logged F1:
    # dict(name='word_s43', kind='part2', checkpoint=ROOT/'results'/'ensemble_probe'/'members'/'word_s43'/'best.pt',
    #      expected=dict(split='checkpoint', threshold=0., policy='original', f1=...)),
]

## 1. Frozen test-like split (build once, then commit `ensemble_split.json`)

The rows that an adversarial classifier ranks most test-like, taken from labelled rows outside every Part 2 partition. They are packed into multi-line posts sized like the test texts, because source rows are single-line and test texts mostly are not. Members trained on more data must exclude these rows (`ensemble_data.testlike_rows`).

In [ ]:
from ensemble_data import build_ensemble_split, read_ensemble_split
if not ENSEMBLE_SPLIT.exists():
    build_ensemble_split(DATA, SPLIT, ENSEMBLE_SPLIT, pool_rows=200000, testlike_rows=2000)
    RESULTS.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ENSEMBLE_SPLIT, RESULTS/'ensemble_split.json')   # the clone is temporary: download and commit it
ensemble_spec = read_ensemble_split(ENSEMBLE_SPLIT, SPLIT)
{k: ensemble_spec[k] for k in ['pool_rows', 'adversarial_auc', 'score_min', 'score_median', 'multirow_share', 'test_newline_share']}

In [ ]:
from ensemble_data import load_docs
SPLITS = ['testlike_select', 'testlike_report', 'labeling', 'checkpoint', 'calibration', 'test']
docs = load_docs(DATA, SPLITS, SPLIT, ENSEMBLE_SPLIT)
{k: len(v) for k, v in docs.items()}

## 2. Probability bundles (one pass per member and split; resumable)

In [ ]:
import torch
from ensemble_members import export_member
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for member in MEMBERS:
    print(member['name'], 'verified F1:', export_member(member, docs, BUNDLES, device))

## 3. Recipe search

In [ ]:
from ensemble import run_search, log_wandb
SEARCH = RESULTS/'search_v1'          # new folder for every change of members or settings
names = [m['name'] for m in MEMBERS]
result, report = run_search(BUNDLES, names, docs, SEARCH, rounds=10, max_size=4)
log_wandb(SEARCH.name, SEARCH)
result['final']

In [ ]:
report.pivot_table(index='recipe', columns='split', values='f1').sort_values('testlike_select', ascending=False)

## 4. Submission and the actual public score

In [ ]:
from ensemble import export_test
final = result['final']
submission = export_test(final, BUNDLES, docs['test'], SEARCH/'submission.csv')
submission

In [ ]:
# After uploading to Kaggle, record the real public score (never a guess):
# from ensemble import record_submission
# record_submission(RESULTS/'leaderboard.csv', final, submission, public_f1=0.0, note='')

## 5. Audit (once, after the recipe is frozen)

In [ ]:
# from ensemble import score_audit
# audit = load_docs(DATA, ['audit'], SPLIT, allow_audit=True)['audit']
# for member in MEMBERS: export_member(member, dict(audit=audit), BUNDLES, device)
# score_audit(final, BUNDLES, audit, SEARCH)